# Lloyds Bank - Dispute Desk Agent
This notebook implements an automated Dispute Desk Agent using **LangChain** and OpenAI:
1. **Dispute Desk Agent Setup**: Built with LangChain tools and agent framework (`create_agent`).
2. **Bank Transactions & Policies Dummy Data**: Transactions (`TRANSACTIONS`), Dispute Policies (`POLICY_CLAUSES`), Reference Answers, and Sample Disputes.
3. **Dispute Resolution**: Invoking the agent to investigate transactions, match eligibility criteria, and approve provisional refunds.
4. **Credit Card Notification Query**: Handling *"I paid my credit card bill , yet I got a message from the bank"* by checking settlement logs and clarifying batch messaging schedules.


In [3]:
import langchain
langchain.__version__


'1.4.3'

In [4]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
print("Environment and API key configured.")


Environment and API key configured.


## 1. Bank Dummy Data
We define the transaction database, Lloyds Bank dispute policy clauses, reference ground truth, and sample customer dispute queries.


In [5]:
# Bank Transactions dummy data
TRANSACTIONS = {
    "TXN-0021": {
        "customer_id": "CUS-0042",
        "merchant": "Techzone Ltd",
        "amount_gbp": 849.00,
        "date": "2024-03-15",
        "channel": "online",
        "status": "disputed",
        "duplicate_detected": False,
    },
    "TXN-0029": {
        "customer_id": "CUS-0017",
        "merchant": "PENTONVALE STORES LTD",
        "amount_gbp": 160.54,
        "date": "2024-04-03",
        "channel": "contactless",
        "status": "disputed",
        "duplicate_detected": False,
    },
    "TXN-0030": {
        "customer_id": "CUS-0088",
        "merchant": "Yewtree Field House",
        "amount_gbp": 500.49,
        "date": "2024-04-12",
        "channel": "chip-and-pin",
        "status": "disputed",
        "duplicate_detected": True,
    },
}

# Bank Dispute Policy Clauses
POLICY_CLAUSES = {
    "MER-13.1": (
        "Goods not received: eligible if merchant has not provided proof of delivery "
        "within 15 calendar days of written request. Raise within 120 days of transaction."
    ),
    "MER-13.3": (
        "Services not rendered: eligible if service was paid for but not delivered. "
        "Evidence of booking confirmation and non-delivery required."
    ),
    "PRC-12.5": (
        "Amount discrepancy: eligible if charged amount differs from authorised amount "
        "by more than £0.50. Provide receipt or authorisation record."
    ),
    "PRC-12.6": (
        "Duplicate transaction: eligible if same purchase was charged more than once."
    ),
}

VALID_REASON_CODES = set(POLICY_CLAUSES.keys())

REFERENCE_ANSWERS = {
    "TXN-0021": {"reason_code": "MER-13.1", "amount_gbp": 849.00},
    "TXN-0029": {"reason_code": "MER-13.1", "amount_gbp": 160.54},
    "TXN-0030": {"reason_code": "PRC-12.6", "amount_gbp": 500.49},
}

SAMPLE_DISPUTES = [
    (
        "TXN-0021",
        "I ordered a laptop from Techzone Ltd on 14 March for £849. "
        "It never arrived and the merchant is not responding. Transaction TXN-0021."
    ),
    (
        "TXN-0029",
        "I was charged £160.54 by Pentonvale Stores Ltd on 3 April "
        "but I have never shopped there. This is fraud. Transaction TXN-0029."
    ),
    (
        "TXN-0030",
        "I stayed at Yewtree Field House but they charged me twice for "
        "the same night. Original receipt shows £250.25 but statement shows £500.49. "
        "Transaction TXN-0030."
    ),
]

# Credit Card Account Records & Notification Logs
CREDIT_CARD_ACCOUNTS = {
    "CUS-0042": {
        "customer_id": "CUS-0042",
        "card_product": "Lloyds Platinum Credit Card (ending 4821)",
        "current_balance_gbp": 0.00,
        "statement_balance_gbp": 250.00,
        "payment_due_date": "2024-04-15",
        "recent_payment": {
            "payment_id": "PAY-88219",
            "amount_gbp": 250.00,
            "payment_date": "2024-04-14",
            "payment_channel": "Faster Payments (Bank App)",
            "settlement_status": "Received & Processed (Account balance updated to £0.00)",
        },
        "recent_automated_messages": [
            {
                "timestamp": "2024-04-14 09:00",
                "channel": "SMS",
                "message": (
                    "Lloyds Bank: Your credit card payment of £250 is due tomorrow. "
                    "If you have already paid in the last 24-48 hours, please disregard this message."
                ),
                "reason": "Automated batch reminder dispatched prior to the 17:00 daily payment reconciliation cycle."
            }
        ]
    }
}

print(f"Loaded {len(TRANSACTIONS)} transactions.")
print(f"Loaded {len(POLICY_CLAUSES)} dispute policies.")
print(f"Loaded {len(CREDIT_CARD_ACCOUNTS)} credit card account records.")


Loaded 3 transactions.
Loaded 4 dispute policies.
Loaded 1 credit card account records.


## 2. Dispute Desk Tools
We build four specialized tools for the agent:
1. `get_transaction_details`: Retrieves transaction data (merchant, amount, date, channel, status).
2. `get_dispute_policies`: Retrieves relevant policy criteria and reason codes.
3. `raise_dispute_case`: Lodges a formal dispute case, applies policy validation, and grants provisional refund.
4. `check_credit_card_payment_status`: Inspects credit card payments, settlement status, and automated bank notification logs.


In [6]:
import json
from langchain_core.tools import tool

@tool
def get_transaction_details(transaction_id: str) -> str:
    """Retrieve details of a bank transaction by ID (e.g. TXN-0021, TXN-0029, TXN-0030)."""
    txn = TRANSACTIONS.get(transaction_id.strip())
    if not txn:
        return f"Transaction {transaction_id} not found in bank records."
    return json.dumps(txn, indent=2)

@tool
def get_dispute_policies(reason_code: str = "") -> str:
    """Retrieve bank dispute policies or check a specific reason code (e.g., MER-13.1, MER-13.3, PRC-12.5, PRC-12.6)."""
    if reason_code and reason_code in POLICY_CLAUSES:
        return f"{reason_code}: {POLICY_CLAUSES[reason_code]}"
    return json.dumps(POLICY_CLAUSES, indent=2)

@tool
def raise_dispute_case(transaction_id: str, reason_code: str, disputed_amount: float, customer_explanation: str) -> str:
    """Formally lodge a dispute claim for a transaction after verifying policy eligibility.
    Requires transaction_id, valid reason_code (e.g. MER-13.1, PRC-12.6), disputed_amount, and customer_explanation."""
    if transaction_id not in TRANSACTIONS:
        return f"Error: Transaction {transaction_id} does not exist in bank records."
    if reason_code not in VALID_REASON_CODES:
        return f"Error: Invalid reason code '{reason_code}'. Must be one of {list(VALID_REASON_CODES)}."
    
    case_id = f"DISP-{transaction_id[-4:]}-RESOLVED"
    record = {
        "case_id": case_id,
        "status": "APPROVED_PROVISIONAL_CREDIT",
        "transaction_id": transaction_id,
        "reason_code": reason_code,
        "policy_rule": POLICY_CLAUSES[reason_code],
        "disputed_amount_gbp": disputed_amount,
        "outcome": f"Dispute validated under {reason_code}. Provisional refund of £{disputed_amount:.2f} credited to customer account."
    }
    return json.dumps(record, indent=2)

@tool
def check_credit_card_payment_status(customer_id: str = "CUS-0042") -> str:
    """Check credit card payment history, cleared balance, and automated notification logs.
    Use this when a customer states they paid their credit card bill but received a message/reminder from the bank."""
    cid = customer_id.strip() if customer_id else "CUS-0042"
    account = CREDIT_CARD_ACCOUNTS.get(cid, CREDIT_CARD_ACCOUNTS.get("CUS-0042"))
    if not account:
        return f"No credit card record found for customer ID '{cid}'."
    return json.dumps(account, indent=2)

dispute_desk_tools = [
    get_transaction_details,
    get_dispute_policies,
    raise_dispute_case,
    check_credit_card_payment_status
]

print(f"Initialized {len(dispute_desk_tools)} tools: {[t.name for t in dispute_desk_tools]}")


Initialized 4 tools: ['get_transaction_details', 'get_dispute_policies', 'raise_dispute_case', 'check_credit_card_payment_status']


## 3. Create the Lloyds Bank Dispute Desk Agent
Using `langchain.agents.create_agent` with `ChatOpenAI(model="gpt-4o-mini", temperature=0)`.
The agent is guided by a system prompt enforcing Lloyds Bank dispute protocol:
- Verify transaction data first.
- Match appropriate policy reason codes.
- Lodge the case and grant provisional credit.
- Provide reassurance and clear explanations for credit card billing and message alerts.


In [7]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

system_prompt = """You are the Lloyds Bank Dispute Desk Agent.
Your role:
1. Assist customers with transaction disputes, billing queries, and payment inquiries.
2. Investigate disputed transactions using `get_transaction_details`.
3. Review eligibility against bank policies using `get_dispute_policies` and determine the exact reason code:
   - MER-13.1: Goods not received.
   - MER-13.3: Services not rendered.
   - PRC-12.5: Amount discrepancy.
   - PRC-12.6: Duplicate transaction.
4. When eligible, lodge the case using `raise_dispute_case` and inform the customer of the provisional refund and case ID.
5. If a customer mentions: 'I paid my credit card bill , yet I got a message from the bank', use `check_credit_card_payment_status`.
   Clearly explain:
   - Payment clearing/settlement timelines (Faster Payments are received promptly, but reconciliation occurs in batches).
   - Automated messages (e.g. reminders) are queued in advance before reconciliation batches run.
   - Reassure the customer of their zero balance and that they may safely disregard the reminder.
Always maintain an empathetic, professional, and clear Lloyds Bank customer service tone."""

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
dispute_agent = create_agent(llm, tools=dispute_desk_tools, system_prompt=system_prompt)
print("Lloyds Bank Dispute Desk Agent created and ready.")


Lloyds Bank Dispute Desk Agent created and ready.


## 4. Invoke the Agent to Solve a Dispute
### Case 1: Transaction TXN-0021 (Goods Not Received)
Customer ordered a laptop for £849.00 from Techzone Ltd which never arrived.


In [8]:
txn_id_1, dispute_text_1 = SAMPLE_DISPUTES[0]
print(f"Customer Query: {dispute_text_1}\n")

result_1 = dispute_agent.invoke({
    "messages": [{"role": "user", "content": dispute_text_1}]
})

print("=" * 60)
print("Agent Response:")
print("=" * 60)
print(result_1["messages"][-1].content)


Customer Query: I ordered a laptop from Techzone Ltd on 14 March for £849. It never arrived and the merchant is not responding. Transaction TXN-0021.

Agent Response:
I've successfully lodged a dispute for your transaction with Techzone Ltd regarding the laptop that never arrived. Here are the details:

- **Transaction ID:** TXN-0021
- **Disputed Amount:** £849.00
- **Reason for Dispute:** Goods not received (MER-13.1)
- **Case ID:** DISP-0021-RESOLVED
- **Outcome:** Your dispute has been validated, and a provisional refund of **£849.00** has been credited to your account.

If you have any further questions or need assistance, please feel free to ask.


### Case 2: Transaction TXN-0030 (Duplicate Transaction)
Customer was charged £500.49 instead of £250.25 (duplicate charge detected).


In [7]:
txn_id_2, dispute_text_2 = SAMPLE_DISPUTES[2]
print(f"Customer Query: {dispute_text_2}\n")

result_2 = dispute_agent.invoke({
    "messages": [{"role": "user", "content": dispute_text_2}]
})

print("=" * 60)
print("Agent Response:")
print("=" * 60)
print(result_2["messages"][-1].content)


Customer Query: I stayed at Yewtree Field House but they charged me twice for the same night. Original receipt shows £250.25 but statement shows £500.49. Transaction TXN-0030.

Agent Response:
I have investigated your transaction with Yewtree Field House and confirmed that a duplicate transaction was detected on transaction TXN-0030.

Here is the resolution for your dispute:

- **Transaction ID:** TXN-0030
- **Merchant:** Yewtree Field House
- **Disputed Amount:** £500.49
- **Reason Code:** PRC-12.6 (Duplicate transaction)
- **Case Reference:** DISP-0030-RESOLVED
- **Status:** Approved Provisional Credit

**Resolution:**
Your dispute has been approved under policy clause **PRC-12.6** (Duplicate transaction). A provisional refund of **£500.49** has been credited back to your account.

Please let us know if there is anything else we can assist you with.


## 5. Handling Query: "I paid my credit card bill , yet I got a message from the bank"
Customers frequently encounter situations where they make a payment towards their credit card bill, but soon after receive an automated reminder or warning message from the bank.
Here, we invoke the agent to investigate customer CUS-0042's account, inspect the payment timeline, and provide a clear, reassuring explanation.


In [8]:
cc_query = "I am customer CUS-0042. I paid my credit card bill , yet I got a message from the bank"
print(f"Customer Query: {cc_query}\n")

result_cc = dispute_agent.invoke({
    "messages": [{"role": "user", "content": cc_query}]
})

print("=" * 60)
print("Agent Response:")
print("=" * 60)
print(result_cc["messages"][-1].content)


Customer Query: I am customer CUS-0042. I paid my credit card bill , yet I got a message from the bank

Agent Response:
Thank you for reaching out, and I completely understand how concerning it can be to receive a payment reminder message after you have already paid your bill.

I have reviewed your Lloyds Platinum Credit Card account (ending 4821):

1. **Payment Received:** We received your payment of **£250.00** on **14 April 2024** via Faster Payments.
2. **Current Balance:** Your payment has been successfully processed and your current balance is **£0.00**.
3. **Why You Received the Message:**
   - The SMS reminder sent on 14 April at 09:00 was an automated batch reminder scheduled prior to the 17:00 daily payment reconciliation cycle.
   - Because the reminder was dispatched before the daily reconciliation batch ran, the message crossed in the system with your payment.

**What You Need to Do:**
You do not need to take any action. Your payment is safely recorded, your account is ful

## 6. Dispute Desk & Credit Card Message Explanation Summary

### Why Customers Receive Messages After Paying Their Credit Card Bill:
1. **Batch Processing vs Real-Time Messaging:**
   - Credit card billing and statement management systems often execute payment reconciliations in scheduled end-of-day batches (e.g. 17:00).
   - Automated notification queues (SMS/Email/Push reminders) are prepared and dispatched earlier in the morning based on balances before the batch reconciliation.
2. **Clearing Times:**
   - Faster Payments typically show in available balance within 2 hours, but statement status updates overnight.
   - Standing orders or external bank transfers can take 1–3 business days.
3. **Best Practices for Customers:**
   - Check the **Lloyds Bank Mobile Banking App** to verify current balance and recent transactions.
   - Verify payment debited from the current account with correct credit card 16-digit number as reference.
   - Ignore payment due alerts if payment was submitted within the last 24–48 hours before the due date.
